# Lab 20-10 · PLLs and Jitter

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import pll

ICP, KV, N = 500e-6, 2 * np.pi * 500e6, 100


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The open loop

*Predict* the crossover and phase margin of Example 20-10.2.

In [2]:
r, c1, c2 = pll.design_third_order(2e6, 10, ICP, KV, N)
print(f"R = {r / 1e3:.2f} kohm, C1 = {c1 * 1e12:.1f} pF, C2 = {c2 * 1e12:.2f} pF")
print("Python:", pll.crossover_and_pm(ICP, KV, N, r, c1, c2))
print("ngspice:", ngspice("loop.cir"))

R = 5.59 kohm, C1 = 45.1 pF, C2 = 5.01 pF
Python: (2000000.0, 54.90319877241541)
ngspice: {'fcx': 2000008.0, 'phx': -2.183353, 'fcm': 2.000008, 'pm': 54.90309}


## Part 2 · Spread b

*Predict* which of b = 4, 10, 20 rings longest after a reference phase step.

In [3]:
t = np.linspace(0, 4e-6, 4001)
plt.figure()
for b in (4, 10, 20):
    r, c1, c2 = pll.design_third_order(2e6, b, ICP, KV, N)
    k = ICP * KV / (2 * np.pi * N)
    tau2 = r * c1 * c2 / (c1 + c2)
    num = [k * r * c1, k]
    den = np.polyadd([(c1 + c2) * tau2, c1 + c2, 0, 0], [0, 0, k * r * c1, k])
    _, y = signal.step((num, den), T=t)
    fc, pm = pll.crossover_and_pm(ICP, KV, N, r, c1, c2)
    plt.plot(t * 1e6, y, label=f"b = {b}, PM = {pm:.1f} deg")
plt.xlabel("time (us)"); plt.ylabel("output phase / final"); plt.legend(); plt.show()

## Part 3 · Noise split

*Predict* which bandwidth (0.1, 0.3, 1, 2 or 5 MHz) gives the least jitter for −150 dBc/Hz reference noise and a VCO at −120 dBc/Hz at 1 MHz. Hint: where does $N^2$ times the reference noise meet the VCO's line?

In [4]:
f = np.logspace(3, 9, 3000)
lref = 10 ** (-150 / 10) * N ** 2
lvco = 10 ** (-120 / 10) * (1e6 / f) ** 2
plt.figure()
for fc in (0.1e6, 0.3e6, 1e6, 2e6, 5e6):
    r, c1, c2 = pll.design_third_order(fc, 10, ICP, KV, N)
    href, hvco = pll.closed_loop(2 * np.pi * f, ICP, KV, N, r, c1, c2)
    tot = lref * np.abs(href / N) ** 2 + lvco * np.abs(hvco) ** 2
    jit = np.sqrt(2 * np.trapezoid(tot, f)) / (2 * np.pi * 5e9)
    print(f"fc = {fc / 1e6:g} MHz: {jit * 1e15:.0f} fs")
    plt.semilogx(f, 10 * np.log10(tot), label=f"fc = {fc / 1e6:g} MHz: {jit * 1e15:.0f} fs")
plt.xlabel("offset (Hz)"); plt.ylabel("dBc/Hz"); plt.legend(); plt.show()
print("crossing of N^2 x reference and VCO lines:", 1e6 * 10 ** ((-120 - (-150 + 20 * np.log10(N))) / 20) / 1e6, "MHz")

fc = 0.1 MHz: 226 fs
fc = 0.3 MHz: 172 fs
fc = 1 MHz: 226 fs
fc = 2 MHz: 309 fs
fc = 5 MHz: 483 fs
crossing of N^2 x reference and VCO lines: 0.31622776601683794 MHz


## Part 4 · Jitter bands

*Predict* how much the jitter of Example 20-10.5 changes between 12 kHz–20 MHz and 10 kHz–100 MHz.

In [5]:
prof_f, prof_l = [1e3, 2e6, 2e9], [-100, -100, -160]
for f1, f2 in ((12e3, 20e6), (10e3, 100e6), (10e3, 1e9)):
    print(f"{f1 / 1e3:g} kHz to {f2 / 1e6:g} MHz: {pll.integrate_jitter(prof_f, prof_l, f1, f2, 5e9) * 1e15:.0f} fs")

12 kHz to 20 MHz: 876 fs
10 kHz to 100 MHz: 895 fs
10 kHz to 1000 MHz: 899 fs


## Part 5 · Fractional-N

*Predict* the spur spacing for 123/500 and the busy fraction's noise at 1 MHz.

In [6]:
fref = 50e6
plt.figure()
for k, mod, lab in ((123, 500, "123/500"), (2 ** 23 + 12345, 2 ** 25, "busy fraction")):
    c = pll.mash_sequence(k, mod, 2 ** 17)
    phi = 2 * np.pi * np.cumsum(c - c.mean()); phi -= phi.mean()
    ff, p = signal.welch(phi, fs=fref, window="blackmanharris", nperseg=2 ** 14, return_onesided=False)
    sel = ff > 0
    plt.semilogx(ff[sel], 10 * np.log10(p[sel]), lw=0.6, label=lab)
    if lab != "123/500":
        i = np.argmin(abs(ff - 1e6))
        print(f"busy fraction at 1 MHz: simulated {10 * np.log10(p[i]):.1f}, formula {10 * np.log10(pll.divider_phase_psd(1e6, fref, 3)):.1f} dBc/Hz")
c = pll.mash_sequence(123, 500, 6000)
print("123/500 period:", next(p for p in range(1, 3000) if np.array_equal(c[2000:4000], c[2000 + p:4000 + p])), "cycles")
plt.semilogx(ff[sel], 10 * np.log10(pll.divider_phase_psd(ff[sel], fref, 3)), "k--", label="formula")
plt.xlabel("offset (Hz)"); plt.ylabel("dBc/Hz"); plt.legend(); plt.show()

busy fraction at 1 MHz: simulated -109.3, formula -107.9 dBc/Hz
123/500 period: 1000 cycles
